# 🎵 Spotify Song Recommender

A content-based recommendation system that suggests songs based on audio features.

---
**Dataset:** [Spotify Tracks Dataset](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset) — 114k tracks with audio features

**Approach:** Cosine similarity on normalized audio features (danceability, energy, valence, etc.)

In [ ]:
# ─── Setup ───
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
print('✅ All imports loaded')

In [ ]:
# ─── Load Data ───
df = pd.read_csv('../data/dataset.csv', index_col=0)
print(f'Shape: {df.shape}')
df.head(3)

---
## 1. Data Exploration

In [ ]:
# Overview
print('── Data Types ──')
print(df.dtypes)
print(f'\n── Missing Values ──')
print(df.isnull().sum()[df.isnull().sum() > 0].to_string() if df.isnull().sum().any() else 'No missing values ✨')
print(f'\n── Duplicates ──')
print(f'{df.duplicated().sum()} duplicate rows')

In [ ]:
# Summary statistics for audio features
audio_cols = ['danceability', 'energy', 'key', 'loudness', 'mode',
              'speechiness', 'acousticness', 'instrumentalness',
              'liveness', 'valence', 'tempo', 'duration_ms']
df[audio_cols].describe()

In [ ]:
# Genre distribution
genre_counts = df['track_genre'].value_counts()
print(f'Unique genres: {len(genre_counts)}')
print(f'Top 15 genres:')
genre_counts.head(15)

In [ ]:
# Top genres — bar plot
plt.figure(figsize=(14, 5))
top_genres = genre_counts.head(20)
sns.barplot(x=top_genres.values, y=top_genres.index, hue=top_genres.index, palette='viridis', legend=False)
plt.title('Top 20 Genres by Track Count', fontsize=16, fontweight='bold')
plt.xlabel('Number of Tracks')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Audio feature distributions
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
feature_names = ['danceability', 'energy', 'valence', 'acousticness',
                 'speechiness', 'instrumentalness', 'liveness', 'loudness',
                 'tempo', 'duration_ms', 'popularity', 'key']

for ax, feat in zip(axes.flat, feature_names):
    sns.histplot(df[feat], bins=50, ax=ax, color='#7d3c98', edgecolor='none')
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('')

plt.suptitle('Distribution of Audio Features', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of audio features
plt.figure(figsize=(12, 9))
corr = df[audio_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Data Preparation

In [ ]:
# Select features for recommendation
feature_cols = ['danceability', 'energy', 'loudness', 'speechiness',
                'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Drop rows with any missing feature values
df_clean = df.dropna(subset=feature_cols).reset_index(drop=True)
print(f'Rows after cleaning: {len(df_clean)} (dropped {len(df) - len(df_clean)})')

In [ ]:
# Normalize features to [0, 1]
scaler = MinMaxScaler()
feature_scaled = scaler.fit_transform(df_clean[feature_cols])

# Build feature matrix
feature_matrix = pd.DataFrame(feature_scaled, columns=feature_cols)
print(f'Feature matrix shape: {feature_matrix.shape}')
feature_matrix.head(3)

---
## 3. Content-Based Recommender

**How it works:** Each song is a vector of audio features. We use `NearestNeighbors` (cosine distance) to find the most similar songs without ever building the full 114k×114k dense matrix — that would need ~97 GiB of RAM. Only the top-k neighbors are computed on the fly.

In [ ]:
# NearestNeighbors instead of full cosine similarity matrix
# A full 114k × 114k dense matrix = ~97 GiB — not feasible.
# NearestNeighbors computes only the top-k on the fly. ~43 MB.
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=50, metric='cosine', algorithm='brute', n_jobs=-1)
nn.fit(feature_matrix.values)
print(f'✅ NearestNeighbors fitted on {feature_matrix.shape[0]} songs with {feature_matrix.shape[1]} features')

In [ ]:
def recommend_by_song(title, artist=None, n=10):
    """
    Recommend songs similar to a given song.
    
    Parameters
    ----------
    title : str — song name to base recommendations on
    artist : str, optional — filter by artist for accuracy
    n : int — number of recommendations to return
    
    Returns
    -------
    DataFrame with recommended songs, similarity scores, and audio features
    """
    # Find matching song
    mask = df_clean['track_name'].str.lower().str.contains(title.lower())
    if artist:
        mask &= df_clean['artists'].str.lower().str.contains(artist.lower())
    
    matches = df_clean[mask]
    if len(matches) == 0:
        print(f'❌ No songs found matching "{title}"')
        return None
    
    idx = matches.index[0]
    song_name = df_clean.loc[idx, 'track_name']
    song_artist = df_clean.loc[idx, 'artists']
    
    # Get nearest neighbors (no full matrix needed)
    query_vec = feature_matrix.values[idx].reshape(1, -1)
    distances, indices = nn.kneighbors(query_vec, n_neighbors=n+1)
    
    # Skip the first (it's the song itself)
    neighbor_indices = indices[0][1:n+1]
    neighbor_scores = [round(1 - d, 4) for d in distances[0][1:n+1]]
    
    result = df_clean.iloc[neighbor_indices].copy()
    result['similarity'] = neighbor_scores
    
    print(f'🎵 Based on: "{song_name}" — {song_artist}\n')
    return result[['track_name', 'artists', 'track_genre', 'similarity',
                   'popularity', 'danceability', 'energy', 'valence']]

In [ ]:
def recommend_by_features(danceability=0.5, energy=0.5, valence=0.5,
                          acousticness=0.3, instrumentalness=0.1, n=10):
    """
    Recommend songs by specifying desired audio feature values.
    Each feature value should be between 0 and 1.
    """
    query = np.array([[danceability, energy, 0, acousticness,
                       instrumentalness, 0, valence, 0, 0]])
    
    # loudness, speechiness, liveness, tempo dummy = 0
    # We need all 9 features in order: danceability, energy, loudness, speechiness,
    # acousticness, instrumentalness, liveness, valence, tempo
    query = np.array([[danceability, energy, 0.5, 0.1,
                       acousticness, instrumentalness, 0.2, valence, 120]])
    query_scaled = scaler.transform(query)
    
    sim_scores = cosine_similarity(query_scaled, feature_matrix)[0]
    top_idx = sim_scores.argsort()[-n:][::-1]
    
    result = df_clean.iloc[top_idx].copy()
    result['similarity'] = [round(sim_scores[i], 4) for i in top_idx]
    
    print(f'🎧 Recommended for your vibe:\n')
    return result[['track_name', 'artists', 'track_genre', 'similarity',
                   'danceability', 'energy', 'valence', 'acousticness']]

---
## 4. Try It — Song-Based Recommendations

In [ ]:
# Example: Find songs similar to "Blinding Lights" by The Weeknd
recommend_by_song('Blinding Lights', artist='Weeknd', n=10)

In [ ]:
# Try another one — pick a song you love!
# recommend_by_song('your favorite song', n=10)

---
## 5. Try It — Vibe-Based Recommendations

No song in mind? Describe your vibe by adjusting the sliders below.

In [ ]:
# High energy, high danceability, high valence = party mode 🎉
recommend_by_features(danceability=0.8, energy=0.8, valence=0.7,
                      acousticness=0.1, instrumentalness=0.1, n=10)

In [ ]:
# Chill acoustic vibe 🍃
recommend_by_features(danceability=0.3, energy=0.2, valence=0.4,
                      acousticness=0.9, instrumentalness=0.2, n=10)

In [ ]:
# Late night instrumental / focus mode 🌙
recommend_by_features(danceability=0.2, energy=0.2, valence=0.3,
                      acousticness=0.6, instrumentalness=0.8, n=10)

---
## 6. Genre-Filtered Recommender

Same content-based approach, but restricted to a specific genre.

In [ ]:
def recommend_in_genre(title, genre, n=10):
    """Recommend songs similar to `title` but only within a specific genre."""
    mask = df_clean['track_name'].str.lower().str.contains(title.lower())
    matches = df_clean[mask]
    if len(matches) == 0:
        print(f'❌ No songs found matching "{title}"')
        return None
    idx = matches.index[0]
    
    # Filter by genre
    genre_mask = df_clean['track_genre'].str.lower() == genre.lower()
    genre_indices = df_clean[genre_mask].index
    
    # Compute similarity on the fly (only for the genre subset — tiny!)
    query_vec = feature_matrix.values[idx].reshape(1, -1)
    target_vecs = feature_matrix.values[genre_indices]
    sim_scores = cosine_similarity(query_vec, target_vecs)[0]
    
    top_ranked = sim_scores.argsort()[-n:][::-1]
    scores = [(genre_indices[i], sim_scores[i]) for i in top_ranked]
    
    result = df_clean.iloc[[i[0] for i in scores]].copy()
    result['similarity'] = [round(i[1], 4) for i in scores]
    
    print(f'🎵 "{df_clean.loc[idx, "track_name"]}" → {genre.title()}\n')
    return result[['track_name', 'artists', 'track_genre', 'similarity', 'popularity']]

In [ ]:
# Find songs similar to a pop track but within jazz
recommend_in_genre('Blinding Lights', 'jazz', n=10)

---
## 7. Popularity-Based Baseline

Sometimes you just want the most popular songs in a genre.

In [ ]:
def popular_in_genre(genre, n=10):
    """Get the most popular tracks in a given genre."""
    result = df_clean[df_clean['track_genre'].str.lower() == genre.lower()] \
        .sort_values('popularity', ascending=False) \
        .head(n)
    if len(result) == 0:
        print(f'❌ Genre "{genre}" not found')
        return None
    print(f'🏆 Top {n} in {genre.title()}\n')
    return result[['track_name', 'artists', 'popularity', 'track_genre']].reset_index(drop=True)

In [ ]:
# Most popular hip-hop tracks
popular_in_genre('hip-hop', n=10)

In [ ]:
# Most popular in any genre you like
# popular_in_genre('rock', n=10)

---
## 8. Summary

### What we built:

| Recommender | How it works | Use case |
|:------------|:-------------|:--------|
| **Content-Based** | Cosine similarity on audio features | "I like this song — find me more like it" |
| **Vibe-Based** | Query by feature values | "I want something danceable and energetic" |
| **Genre-Filtered** | Similarity within a genre | "Find me jazz that sounds like this pop song" |
| **Popularity** | Sort by popularity | "What's hot in this genre?" |

### Key insights:
- The dataset has **114k tracks** across **114 genres** with **21 features** each
- Audio features like **energy**, **danceability**, and **valence** are the strongest signals for similarity
- Content-based filtering works well when you have rich feature data — no need for user history
- **Next steps:** Could extend with collaborative filtering (user ratings) or hybrid approaches

### Try it yourself:
- Run `recommend_by_song('Your Song')` in a cell above
- Or use `recommend_by_features()` to find songs by vibe
- Or `popular_in_genre('genre')` for the hits

---
*Built as a learning project. Data from [Spotify Tracks Dataset](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset) on Hugging Face.*